In [3]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# ─────────────────────────────────────────────────────────────────────
# STEP 1 — Load your hourly data
# Save your 8184 values to a CSV file called Load_data.csv (one value
# per row, no header), then run this script.
# ─────────────────────────────────────────────────────────────────────
df_raw = pd.read_csv(r"C:\Users\wb590499\Documents\Projects\EPM\epm\input\data_zim\Load_data.csv", header=None, names=["load"])
raw_values = df_raw["load"].values

# ─────────────────────────────────────────────────────────────────────
# STEP 2 — Reshape into daily profiles (341 days x 24 hours)
# ─────────────────────────────────────────────────────────────────────
n_hours = len(raw_values)
n_days  = n_hours // 24
assert n_days * 24 == n_hours, "Data length must be divisible by 24"

daily_profiles = raw_values[:n_days * 24].reshape(n_days, 24)

# ─────────────────────────────────────────────────────────────────────
# STEP 3 — Define seasons (adjust boundaries to match your calendar)
# ─────────────────────────────────────────────────────────────────────
season_bounds = {
    "Q1": (0,   90),
    "Q2": (90,  181),
    "Q3": (181, 272),
    "Q4": (272, n_days)
}

# ─────────────────────────────────────────────────────────────────────
# STEP 4 — K-means clustering within each season
# ─────────────────────────────────────────────────────────────────────
K           = 4
RANDOM_SEED = 42

results = []

for season, (start, end) in season_bounds.items():
    season_profiles = daily_profiles[start:end]
    km = KMeans(n_clusters=K, random_state=RANDOM_SEED, n_init=20)
    km.fit(season_profiles)

    labels    = km.labels_
    centroids = km.cluster_centers_

    # Sort clusters by mean load (ascending): d1=low, d4=high
    order = np.argsort(centroids.mean(axis=1))

    for rank, cluster_idx in enumerate(order):
        weight   = int(np.sum(labels == cluster_idx))
        centroid = centroids[cluster_idx]

        row = {"q": season, "d": f"d{rank + 1}", "weight": weight}
        for h in range(24):
            row[f"t{h + 1}"] = round(centroid[h], 1)
        results.append(row)

# ─────────────────────────────────────────────────────────────────────
# STEP 5 — Build output tables
# ─────────────────────────────────────────────────────────────────────
cols   = ["q", "d", "weight"] + [f"t{h}" for h in range(1, 25)]
df_out = pd.DataFrame(results, columns=cols)

# Weight table: same weight repeated across all 24 hour columns
df_weights = df_out[["q", "d"]].copy()
for h in range(1, 25):
    df_weights[f"t{h}"] = df_out["weight"]

# ─────────────────────────────────────────────────────────────────────
# STEP 6 — Print results
# ─────────────────────────────────────────────────────────────────────
print("\n=== Representative Day Profiles (centroid values) ===\n")
print(df_out.to_string(index=False))

print("\n=== Weight Table ===\n")
print(df_weights.to_string(index=False))

print("\n=== Verification: weight sums per season ===")
for season, (start, end) in season_bounds.items():
    s = df_out[df_out["q"] == season]["weight"].sum()
    expected = end - start
    status = "OK" if s == expected else "MISMATCH"
    print(f"  {season}: {s} / {expected} days  [{status}]")

# ─────────────────────────────────────────────────────────────────────
# STEP 7 — Export to CSV
# ─────────────────────────────────────────────────────────────────────
df_out.to_csv("representative_days_profiles.csv", index=False)
df_weights.to_csv("representative_days_weights.csv", index=False)
print("\nSaved: representative_days_profiles.csv")
print("Saved: representative_days_weights.csv")

#This document was generated by mAI.


c:\Users\wb590499\.conda\envs\epm_env\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\wb590499\.conda\envs\epm_env\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\wb590499\.conda\envs\epm_env\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\wb590499\.conda\envs\epm_env\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarn


=== Representative Day Profiles (centroid values) ===

 q  d  weight     t1     t2     t3     t4     t5     t6     t7     t8     t9    t10    t11    t12    t13    t14    t15    t16    t17    t18    t19    t20    t21    t22    t23    t24
Q1 d1       2 1092.5 1071.0 1056.5 1064.0 1194.5 1324.5 1157.0 1073.5 1096.5 1072.0 1046.0 1014.0 1005.0  991.5 1013.5 1048.5 1041.0 1090.0    0.0    0.0    0.0    0.0    0.0    0.0
Q1 d2       1    0.0    0.0    0.0    0.0    0.0    0.0 1298.0 1278.0 1326.0 1322.0 1307.0 1300.0 1262.0 1231.0 1253.0 1246.0 1248.0 1065.0 1055.0 1213.0 1250.0 1244.0 1285.0 1189.0
Q1 d3      37 1160.4 1146.4 1133.8 1141.9 1189.3 1125.3 1011.9  963.8  970.5  950.9  930.2  919.1  906.8  902.5  906.6  917.7  926.7  937.6  960.6  992.4 1044.0 1128.9 1194.2 1198.2
Q1 d4      50 1053.0 1037.0 1028.0 1030.8 1089.3 1187.8 1179.9 1162.7 1183.9 1178.0 1161.6 1146.3 1136.8 1131.2 1137.5 1162.2 1195.3 1218.7 1242.2 1233.3 1173.2 1162.3 1121.0 1075.3
Q2 d1       1 1213.0 1222.0 1232.0